# Ligat HaAl round-by-round ranking changes

- **Regular**: cumulative tables from TM `matches_*_ligat_haal_transfermarkt[_dated].csv`.
- **Split era**: subgroup pools from TM tracking CSVs; carry points halved before mini-leagues. **2009/10 and 2010/11** use **truncation `pts // 2` (documented)**; other seasons use `int(round(pts / 2))`.
- **Playoffs**: TM rounds after `regular_last_round` when present; else Betexplorer `scraped_betexplorer/championship_*.csv` and `relegation_*.csv`.
- **Cell 1 (export)**: with `SEASONS = None`, processes **all** seasons that have `regular_season_tracking_*.csv` and writes full CSVs.
- **Cell 2 (optional)**: prints long text reports for a **small example list** only — not the full league list.
- Output: `notebooks/data/processed/rank_dynamics/`; each block lists rank moves or **No ranking changes in this round**.

In [2]:
from __future__ import annotations
import sys
from pathlib import Path

# Resolve `notebooks/` root whether cwd is `notebooks/` or a parent repo folder
cwd = Path.cwd().resolve()

def resolve_notebooks_root() -> Path:
    for base in [cwd, *cwd.parents]:
        if (base / "league_dynamics" / "rank_tracking.py").is_file():
            return base
        nb = base / "notebooks"
        if (nb / "league_dynamics" / "rank_tracking.py").is_file():
            return nb
    raise FileNotFoundError(
        "Could not find notebooks/league_dynamics/rank_tracking.py "
        "(set kernel cwd to the directory that contains `league_dynamics/` or `notebooks/league_dynamics/`)."
    )

ROOT = resolve_notebooks_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from league_dynamics import rank_tracking as rt

OUT = ROOT / "data" / "processed" / "rank_dynamics"
OUT.mkdir(parents=True, exist_ok=True)

SEASONS = None  # None = every season that has regular_season_tracking_*.csv; or e.g. ["2015/16"]
changes_df, meta_df = rt.run_all_seasons(SEASONS)

changes_df.to_csv(OUT / "rank_changes_export.csv", index=False, encoding="utf-8-sig")
meta_df.to_csv(OUT / "rank_dynamics_season_meta.csv", index=False, encoding="utf-8-sig")
print("Wrote:", OUT, "| seasons in export:", meta_df.shape[0])

try:
    display(meta_df.head(15))
    display(changes_df.head(40))
except NameError:
    print(meta_df.head())
    print(changes_df.head())

Wrote: C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\processed\rank_dynamics | seasons in export: 19


,season,explicit_kizuz,regular_last_round,playoff_source,warnings,notes
0,2006/07,False,33,none,[],Legacy era: no playoff split tracked for this ...
1,2007/08,False,33,none,[],Legacy era: no playoff split tracked for this ...
2,2008/09,False,33,none,[],Legacy era: no playoff split tracked for this ...
3,2009/10,True,30,betexplorer,[],Subgroup tables after the split: ranks are wit...
4,2010/11,True,30,betexplorer,[],Subgroup tables after the split: ranks are wit...
5,2011/12,False,30,betexplorer,[],Subgroup tables after the split: ranks are wit...
6,2012/13,False,26,betexplorer,[],Subgroup tables after the split: ranks are wit...
7,2013/14,False,26,betexplorer,[],Subgroup tables after the split: ranks are wit...
8,2014/15,False,26,betexplorer,[],Subgroup tables after the split: ranks are wit...
9,2015/16,False,26,betexplorer,[],Subgroup tables after the split: ranks are wit...


,season,stage,segment_label,team,prev_rank,new_rank
0,2006/07,regular,Regular: after round 2 vs after round 1,B. Jerusalem,3,2
1,2006/07,regular,Regular: after round 2 vs after round 1,Bnei Yehuda,5,8
2,2006/07,regular,Regular: after round 2 vs after round 1,FC Ashdod,4,6
3,2006/07,regular,Regular: after round 2 vs after round 1,H. Petah Tikva,12,10
4,2006/07,regular,Regular: after round 2 vs after round 1,Hakoah Amidar,7,5
5,2006/07,regular,Regular: after round 2 vs after round 1,Hapoel Tel Aviv,6,9
6,2006/07,regular,Regular: after round 2 vs after round 1,M. Petah Tikva,8,3
7,2006/07,regular,Regular: after round 2 vs after round 1,M. Tel Aviv,9,11
8,2006/07,regular,Regular: after round 2 vs after round 1,Maccabi Haifa,11,7
9,2006/07,regular,Regular: after round 2 vs after round 1,Maccabi Herzlya,10,12


In [3]:
# Optional: readable console sample only (three seasons shown as examples — full universe is Cell 1)
SAMPLE_PRINT = ["2009/10", "2010/11", "2015/16"]
for sea in SAMPLE_PRINT:
    notes, blocks, val = rt.analyze_season(sea)
    print(rt.format_season_report(notes, blocks, val))
    print()


Season 2009/10:
  Notes:
    - Points halved before playoff groups: True
    - SPECIAL (2009/10 or 2010/11): חוק הקיזוז — carry points truncated with integer division pts // 2
    - Regular season ended after TM round 30
    - Playoff fixture source where applicable: betexplorer
    - Subgroup tables after the split: ranks are within the championship or relegation cohort only (not comparable to pre-split global rank 1–N).
    - Points from the regular season are halved for playoff carry points; goal difference from the regular phase is carried for tie-breaks within each mini-league.
    - חוק הקיזוז seasons (2009/10, 2010/11): carry points halved with integer truncation (pts // 2).
  [regular] Regular: after round 2 vs after round 1
    B. Jerusalem: rank 9 -> 11
    Bnei Sakhnin: rank 15 -> 14
    FC Ashdod: rank 16 -> 7
    H. Beer Sheva: rank 3 -> 4
    H. Petah Tikva: rank 4 -> 6
    H. Ramat Gan: rank 7 -> 8
    Hapoel Acre: rank 11 -> 12
    Hapoel Raanana: rank 12 -> 15
    M. A